# Анализ данных интернет-магазина

In [ ]:
!pip install psycopg2-binary pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 2.8 MB/s eta 0:00:0000:0100:010m


In [ ]:
import psycopg2
import pandas as pd
import os

conn = psycopg2.connect(
    host="postgres",
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)


query = "SELECT * FROM orders"
df = pd.read_sql(query, conn)
df.head()


/tmp/ipykernel_100/1710231968.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,product_name,category,price,quantity,city,created_at
0,1,Phone,Electronics,500.0,3,Moscow,2026-01-20 13:47:00.144759
1,2,Laptop,Electronics,1000.0,5,Saint Petersburg,2026-01-20 13:47:01.155368
2,3,Phone,Electronics,500.0,5,Kazan,2026-01-20 13:47:02.158013
3,4,Laptop,Electronics,1000.0,1,Novosibirsk,2026-01-20 13:47:03.162060
4,5,Book,Books,15.0,2,Kazan,2026-01-20 13:47:04.172685


## Подготовка данных

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['total_price'] = df['price'] * df['quantity']

## Общая статистика магазина

In [ ]:
total_orders = len(df)
total_revenue = df['total_price'].sum()

print(f"Total orders: {total_orders}")
print(f"Total revenue: {total_revenue:.2f}")

## Выручка по категориям

In [ ]:
revenue_by_category = (
    df.groupby('category')['total_price']
      .sum()
      .sort_values(ascending=False)
)

revenue_by_category

In [ ]:
import matplotlib.pyplot as plt

revenue_by_category.plot(kind='bar', figsize=(8,5))
plt.title("Revenue by Category")
plt.ylabel("Revenue")
plt.xlabel("Category")
plt.tight_layout()
plt.show()

## Количество заказов по категориям

In [ ]:
orders_by_category = df['category'].value_counts()
orders_by_category

In [ ]:
orders_by_category.plot(kind='bar', figsize=(8,5))
plt.title("Number of Orders by Category")
plt.ylabel("Orders count")
plt.xlabel("Category")
plt.tight_layout()
plt.show()

category
Books             30.0
Electronics    14500.0
Fashion          400.0
Name: total_price, dtype: float64

## Средний чек по категориям

In [ ]:
avg_order_value = (
    df.groupby('category')['total_price']
      .mean()
      .sort_values(ascending=False)
)

avg_order_value

## Динамика заказов во времени

In [ ]:
orders_over_time = (
    df.set_index('created_at')
      .resample('1min')
      .size()
)

orders_over_time.head()

In [ ]:
orders_over_time.plot(figsize=(10,5))
plt.title("Orders Over Time (per minute)")
plt.ylabel("Orders count")
plt.xlabel("Time")
plt.tight_layout()
plt.show()